<a href="https://colab.research.google.com/github/PrabhuRajendhran/100-days-of-inference/blob/main/vLLM_setup_test_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Kill any hidden background processes trying to run the old setup
!pkill -f vllm

# 2. Forcefully remove the incompatible build and clean the pip cache
!pip uninstall -y vllm vllm-flash-attn
!pip cache purge

# 3. Install a stable vLLM version compiled for CUDA 12
!pip install vllm==0.19.1 --extra-index-url https://pytorch.org

Found existing installation: vllm 0.19.1
Uninstalling vllm-0.19.1:
  Successfully uninstalled vllm-0.19.1
Files removed: 118
Looking in indexes: https://pypi.org/simple, https://pytorch.org
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.1/433.1 MB 1.9 MB/s eta 0:00:00


In [19]:
!pip install transformers accelerate

In [2]:
!nohup python3 -m vllm.entrypoints.openai.api_server \
    --model Qwen/Qwen2.5-0.5B-Instruct \
    --port 8000 \
    --dtype half \
    --gpu-memory-utilization 0.8 \
    > vllm.log 2>&1 &

In [15]:
!tail -n 10 vllm.log

(APIServer pid=9172) INFO 06-23 03:33:47 [launcher.py:46] Route: /v1/messages, Methods: POST
(APIServer pid=9172) INFO 06-23 03:33:47 [launcher.py:46] Route: /v1/messages/count_tokens, Methods: POST
(APIServer pid=9172) INFO 06-23 03:33:47 [launcher.py:46] Route: /inference/v1/generate, Methods: POST
(APIServer pid=9172) INFO 06-23 03:33:47 [launcher.py:46] Route: /scale_elastic_ep, Methods: POST
(APIServer pid=9172) INFO 06-23 03:33:47 [launcher.py:46] Route: /is_scaling_elastic_ep, Methods: POST
(APIServer pid=9172) INFO 06-23 03:33:47 [launcher.py:46] Route: /v1/chat/completions/render, Methods: POST
(APIServer pid=9172) INFO 06-23 03:33:47 [launcher.py:46] Route: /v1/completions/render, Methods: POST
(APIServer pid=9172) INFO:     Started server process [9172]
(APIServer pid=9172) INFO:     Waiting for application startup.
(APIServer pid=9172) INFO:     Application startup complete.


In [16]:
!ps aux | grep vllm

root        9172  7.1  9.9 8349436 1324412 ?     Sl   03:29   0:19 python3 -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-0.5B-Instruct --port 8000 --dtype half --gpu-memory-utilization 0.8
root       10801  0.0  0.0   7376  3552 ?        S    03:33   0:00 /bin/bash -c ps aux | grep vllm
root       10803  0.0  0.0   6484  2416 ?        S    03:33   0:00 grep vllm


In [17]:
!curl http://localhost:8000/v1/models

{"object":"list","data":[{"id":"Qwen/Qwen2.5-0.5B-Instruct","object":"model","created":1782185643,"owned_by":"vllm","root":"Qwen/Qwen2.5-0.5B-Instruct","parent":null,"max_model_len":32768,"permission":[{"id":"modelperm-b9d0dcd5f3d7739d","object":"model_permission","created":1782185643,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]}]}

In [20]:
import json
import urllib.request
import time

url = "http://localhost:8000/v1/chat/completions"
payload = {
    "model": "Qwen/Qwen2.5-0.5B-Instruct",
    "messages": [{"role": "user", "content": "Write a long essay detailing the history of computing sciences."}],
    "temperature": 0.2,
    "max_tokens": 512
}

req = urllib.request.Request(url, data=json.dumps(payload).encode("utf-8"), headers={"Content-Type": "application/json"})

start_time = time.time()
try:
    with urllib.request.urlopen(req) as response:
        result = json.loads(response.read().decode("utf-8"))
    end_time = time.time()

    # Calculate performance metrics
    total_time = end_time - start_time
    tokens_generated = result["usage"]["completion_tokens"]
    tps = tokens_generated / total_time

    print(f"=== vLLM Benchmark Result ===")
    print(f"Tokens Generated: {tokens_generated}")
    print(f"Total Time: {total_time:.2f} seconds")
    print(f"Speed: {tps:.2f} tokens/second")
except Exception as e:
    print(f"Benchmark failed: {e}")

=== vLLM Benchmark Result ===
Tokens Generated: 512
Total Time: 3.41 seconds
Speed: 150.11 tokens/second


In [21]:
!pkill -f vllm

In [22]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import time

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

# Load model natively in FP16 on the T4 GPU
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="cuda")

prompt = "Write a long essay detailing the history of computing sciences."
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# Warmup run to initialize CUDA kernels accurately
_ = model.generate(**inputs, max_new_tokens=10, do_sample=False)

# Actual Benchmark Run
start_time = time.time()
outputs = model.generate(**inputs, max_new_tokens=512, do_sample=False)
end_time = time.time()

# Calculate performance metrics
total_time = end_time - start_time
input_len = inputs.input_ids.shape[1]
tokens_generated = outputs.shape[1] - input_len
tps = tokens_generated / total_time

print(f"=== Vanilla Transformers Benchmark Result ===")
print(f"Tokens Generated: {tokens_generated}")
print(f"Total Time: {total_time:.2f} seconds")
print(f"Speed: {tps:.2f} tokens/second")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

=== Vanilla Transformers Benchmark Result ===
Tokens Generated: 512
Total Time: 21.11 seconds
Speed: 24.26 tokens/second
